# Does learning a temporal order help you dig it out of noise?

The third experiment in this repository. It sits on top of the single-interval yes/no task and
adds three phases: a pre-test, a block of extra exposure to **one** of two sequences, and a
post-test. The question is whether the extra exposure helps that sequence specifically.

The estimand, per participant, is a difference in differences:

$$D_i = (d'_{\text{post,trained}} - d'_{\text{pre,trained}}) - (d'_{\text{post,untrained}} - d'_{\text{pre,untrained}})$$

A positive $D_i$ would say exposure helped *that sequence* beyond general practice. It would
not show neural pre-activation, binding circuitry, implicit learning, or that attention was
uninvolved — the exposure block here is the same attended yes/no task, so it is practice with a
particular sequence and that is all it is.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/notebooks/SeqSFG_exposure_playground.ipynb)

**Runtime → Run all**, CPU, a couple of minutes. Headphones, fixed volume, nothing autoplays.
Everything is computed live from commit `8f1f2c91ba1b`.

In [ ]:
#@title Setup
import sys, subprocess, importlib.util, json
from pathlib import Path
SOURCE_REF='8f1f2c91ba1b93ed5c898660c9b645caf62f133a'
REPO_URL='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
IN_COLAB=bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
if IN_COLAB:
    ROOT=Path('/content')/('seqsfg-exposure-'+SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/exposure.py').exists()),None)
    if ROOT is None: raise RuntimeError('Run from inside the repository, or open in Colab.')
missing=[m for m in ['numpy','scipy','matplotlib'] if importlib.util.find_spec(m) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
sys.path.insert(0,str(ROOT))

import numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
from seqsfg.config import validate
from seqsfg.stimulus import render_interval, FIGURE
from seqsfg import exposure as X, yesno, measure as M, verify as V

cfg, ecfg = X.load_preset(ROOT/'exposure_pilot.json')
X.check(cfg, ecfg)
D = validate(cfg); SR = cfg.sample_rate
P, Q, OV = X.choose_orders(cfg.n_components, ecfg.sequence_seed)
S = X.figure_set_for(cfg, D, ecfg)
ORDERS = {'P': P, 'Q': Q}
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.25,'font.size':9})
INK={'P':'#c1272d','Q':'#1b6ca8','bg':'#b9bec6'}

def trial(order, seed, step, present):
    return X.build_trial(cfg, D, S, order, seed, step, present, ecfg.absent_class)

def play(x):
    lead=np.zeros(cfg.ms_to_samples(cfg.lead_silence_ms),dtype=np.float32)
    return Audio(np.clip(np.concatenate([lead,x]),-1,1), rate=SR, normalize=False)

print('commit        ', SOURCE_REF[:12])
print('config hash   ', cfg.hash(), '   exposure hash', ecfg.hash())
print('figure set    ', S.tolist(), '->', np.round(D.channel_freqs_hz[S]).astype(int).tolist(), 'Hz')
print('tested delays ', list(ecfg.test_steps_ms), 'ms   exposure delays', list(ecfg.exposure_step_list), 'ms')

## 1. Two orders over one set of pitches

P and Q use **the same seven frequencies**. The only thing that separates them is *when* each
component starts inside an element. That is deliberate: if they used different pitches, a
trained-versus-untrained difference could be about the pitches.

The pair is chosen to share as little as the 5040 permutations of seven components allow. The
strictly ascending and strictly descending orders are excluded from both — a monotone contour is
a frequency sweep, a different kind of object, and making one of the pair a sweep would stop them
being exchangeable.

In [ ]:
rows = "\n".join(f"| `{k}` | {v} |" for k,v in OV.items())
display(Markdown(f"**P** onset slots `{list(P)}`, heard order `{X.heard_order(P)}`  \n"
                 f"**Q** onset slots `{list(Q)}`, heard order `{X.heard_order(Q)}`\n\n"
                 f"| what they still share | |\n|---|---|\n{rows}"))

fig,axes=plt.subplots(1,2,figsize=(11,3.4),sharey=True)
oct_=np.log2(D.channel_freqs_hz)
for ax,(k,o) in zip(axes,ORDERS.items()):
    slots=np.asarray(o); y=oct_[S]
    ax.scatter(slots*7.0, y, s=90, marker='_', linewidths=3, color=INK[k])
    for i,(sl,yy) in enumerate(zip(slots,y)):
        ax.annotate(str(i), (sl*7.0, yy), textcoords='offset points', xytext=(6,4), fontsize=7)
    seq=X.heard_order(o)
    ax.plot([np.where(slots==s)[0][0]*7.0 for s in range(len(slots))],
            [y[seq[s]] for s in range(len(slots))], lw=0.8, alpha=0.5, color=INK[k])
    ax.set_title(f'order {k}   (7 ms step shown)'); ax.set_xlabel('onset within the element (ms)')
ticks=[250,1000,4000]
axes[0].set_yticks(np.log2(ticks)); axes[0].set_yticklabels([str(t) for t in ticks])
axes[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()

Zero shared directed transitions means no ordered pair of components that follows in P also
follows in Q — counting the wrap from the last component of one repetition into the first of the
next, because the element repeats and that transition is on offer to learn too.

That is not the same as independence, and the table above is the residual: shared unordered
adjacencies, components landing in the same slot, and the rank correlation of the two heard
sequences. Those are reported rather than waved away.

## 2. What the two orders sound like

First on their own, with the background stripped out. **This is an illustration only** — the
experiment never presents the target in isolation, and the exposure phase in particular uses
exactly the configured background and level. Removing the background here is a teaching aid, not
part of the protocol.

In [ ]:
from seqsfg.stimulus import BACKGROUND
def figure_only(iv):
    m = iv.kind==FIGURE
    out = iv.copy()
    for a in ('onset','channel','phase','kind','element','component'):
        setattr(out,a,getattr(iv,a)[m])
    return render_interval(cfg,out,D)
for k,o in ORDERS.items():
    display(Markdown(f'**order {k}, figure alone (illustration, not the stimulus)**'))
    display(play(figure_only(trial(o, 11, 7.0, True))))

Now the actual stimulus: the full mixture, at the 7 ms delay. Four sounds — each order, present
and absent.

In [ ]:
for k,o in ORDERS.items():
    for present in (True, False):
        display(Markdown(f'**order {k} — figure {"PRESENT" if present else "ABSENT"}** '
                         f'(the answer is "{"yes" if present else "no"}")'))
        display(play(render_interval(cfg, trial(o, 21 if present else 22, 7.0, present), D)))

## 3. What "trained-designated absent" means

This is the part that has to be right, so it is worth being explicit.

A trial's **designation** is which order governs whatever is time-aligned — and it is the same on
yes and no trials:

| | aligned set | answer |
|---|---|---|
| P-designated, present | the fixed set S, components starting in order **P**, every element | yes |
| P-designated, absent | a **fresh band** each element, components starting in order **P** | no |

So within a designation the onset order is identical on yes and no trials. **Order carries no
information about the answer** and cannot be a shortcut. What differs is only whether the
frequency set comes back — which is what detection means here — and each designation has its own
false-alarm reference measured under its own order.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,3.8),sharey=True)
for ax,present in zip(axes,(True,False)):
    iv=trial(P, 31, 14.0, present)
    t=iv.onset*cfg.grid_ms/1000.0; y=oct_[iv.channel]; m=iv.kind==FIGURE
    ax.scatter(t[~m],y[~m],s=5,c=INK['bg'],marker='s',linewidths=0)
    ax.scatter(t[m],y[m],s=20,c=INK['P'],marker='s',linewidths=0)
    ax.set_xlim(0.3,2.0); ax.set_xlabel('time (s)')
    ax.set_title(f'P-designated, {"PRESENT — the same band returns" if present else "ABSENT — a new band each time"}')
axes[0].set_yticks(np.log2(ticks)); axes[0].set_yticklabels([str(t) for t in ticks])
axes[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()

## 4. At zero delay the two orders are the same sound

With no onset separation the components are simultaneous, so P and Q are literally the same
waveform. Step 0 is kept in the design as a shared practice reference and as a manipulation
check — if trained and untrained differ there, something is wrong with the labelling, not with
the listener. **D is never computed at step 0.**

In [ ]:
a = render_interval(cfg, trial(P, 99, 0.0, True), D)
b = render_interval(cfg, trial(Q, 99, 0.0, True), D)
print('identical waveform at step 0:', np.array_equal(a,b), '   max |difference| =', np.abs(a-b).max())
c = render_interval(cfg, trial(P, 99, 14.0, True), D)
e_ = render_interval(cfg, trial(Q, 99, 14.0, True), D)
print('identical waveform at step 14:', np.array_equal(c,e_), '  max |difference| =', round(float(np.abs(c-e_).max()),4))
display(Markdown('**step 0, order P**')); display(play(a))
display(Markdown('**step 0, order Q — the same file**')); display(play(b))

## 5. The three phases

In [ ]:
dz = X.make_design(cfg, ecfg, 'DEMO01', 1)
est = X.duration_estimate(cfg, ecfg)
print(f"participant DEMO01   trained: {dz['trained']}   comparison: {dz['untrained']}")
for ph in ('practice','pre','exposure','post'):
    r=dz['phases'][ph]
    seqs={s:sum(1 for x in r if x['sequence']==s) for s in ('P','Q')}
    print(f"  {ph:<9}{len(r):>4} trials   present {sum(x['present'] for x in r):>3}   P {seqs['P']:>3}  Q {seqs['Q']:>3}")
print(f"  estimated {est['minutes']:.0f} minutes at {est['trial_seconds']:.1f} s per trial plus breaks")
print(f"  unique stimulus seeds: {len({x['seed'] for ph in dz['phases'].values() for x in ph})} "
      f"of {sum(len(v) for v in dz['phases'].values())}  <- fresh acoustics everywhere")

Only the frequency set and the two orders persist across phases. Every trial draws its own
background and its own element timing, so a post-test gain cannot be recognition of a remembered
waveform.

Note what the counts say about the honest limitation: **both** sequences are presented during
testing. Only one gets the extra exposure block. The trained sequence therefore ends the session
with more total presentations, and that is logged per trial (`cum_seq_present`, `cum_seq_any`)
rather than hidden.

## 6. The audit that matters for D

If any acoustic measure could tell P-designated trials from Q-designated ones, a
trained-minus-untrained difference would not need learning to explain it. This is the check the
estimand rests on, run live at the delay D is computed at.

In [ ]:
ref=M.single_tone_reference(cfg,D,V.WIN_MS,V.HOP_MS)
for step in [s for s in ecfg.test_steps_ms if s>0][:1]:
    for cls,present in (('present',True),('absent',False)):
        names,A = X._features(cfg,D,ecfg,S,P,step,present,40,4001,ref)
        _,B     = X._features(cfg,D,ecfg,S,Q,step,present,40,4002,ref)
        r = yesno.feature_separation(names,A,B,4000,7)
        l = yesno.learnt_observer(A,B)
        print(f"step {step:g} ms, {cls:>7}:  P vs Q  permutation p = {r['p_value']:.3f}   "
              f"largest feature d' = {r['observed_max']:.2f}   learnt observer {l['pc']*100:.0f}% correct")

The full audit (`seqsfg exposure-verify`) runs this at every delay, on both designations, with a
Holm correction across its own rows, and uses the step-0 rows as a null check — at zero delay the
two designations are the *same* construction, so whatever the audit says there is its own
false-positive rate, measured rather than assumed. The shipped report is
`verification/exposure_audit_report.txt`.

Its one real finding: at step 0 there is a small non-binding route (a blind classifier reaches
about 59%), driven by the spread rather than the mean of a per-channel timing statistic. It does
not appear at the nonzero delays. Since D is a difference of differences at nonzero delays, a cue
that is common to both designations and both phases cancels exactly.

## 7. Why the contrast is a difference in differences

Because practice alone will move d′. The estimand subtracts it. Here is that arithmetic on
made-up counts, to show what it does and does not remove.

In [ ]:
def demo(label, rates):
    rows=[]
    for (phase,role),(h,f) in rates.items():
        for i in range(20):
            rows.append({'phase':phase,'role':role,'step_ms':'7.0','present':'1','response':'y' if i<round(h*20) else 'n'})
            rows.append({'phase':phase,'role':role,'step_ms':'7.0','present':'0','response':'y' if i<round(f*20) else 'n'})
    r=X.difference_in_differences(rows, X.ExposureConfig(test_steps_ms=(0.0,7.0), n_boot=1500))['per_delay'][7.0]
    print(f"{label:<46} trained gain {r['gain_trained']:+.2f}   comparison gain "
          f"{r['gain_untrained']:+.2f}   D = {r['D']:+.2f}  [{r['ci'][0]:+.2f}, {r['ci'][1]:+.2f}]")

demo('both sequences improve equally (practice)',
     {('pre','trained'):(0.60,0.35),('pre','untrained'):(0.60,0.35),
      ('post','trained'):(0.85,0.20),('post','untrained'):(0.85,0.20)})
demo('only the trained sequence improves',
     {('pre','trained'):(0.60,0.35),('pre','untrained'):(0.60,0.35),
      ('post','trained'):(0.85,0.20),('post','untrained'):(0.60,0.35)})
demo('nothing changes',
     {('pre','trained'):(0.70,0.30),('pre','untrained'):(0.70,0.30),
      ('post','trained'):(0.70,0.30),('post','untrained'):(0.70,0.30)})

General practice cancels; a sequence-specific gain survives. The interval comes from resampling
all eight counts together on every bootstrap draw, not from subtracting the endpoints of four
separately computed d′ intervals — which would be a different and wrong quantity.

## 8. What this design cannot tell you

- **Both sequences are heard during testing.** Only one gets the extra block, so the contrast is
  *more* exposure versus *less*, not exposure versus none. The cumulative counts are logged.
- **The exposure block is the same attended task.** Call it practice with a particular sequence.
  It does not isolate implicit learning and it does not exclude attention.
- **A behavioural delay effect is not an STDP window.** The onset step is an acoustic parameter;
  it is not a synaptic time constant, and the four timescales here — onset step, element
  duration, within-trial repetition, across-phase exposure — are different things.
- **The pilot preset is small.** Eight trials per cell means each d′ is noisy; the delay-specific
  intervals are wide and honestly so. It is sized to be run, not to be decisive.
- **`trained: auto` balances only in expectation.** For a real sample, assign from a
  counterbalancing list and check the balance line the analysis prints.

```
seqsfg exposure-design  --preset exposure_pilot.json --code P01
seqsfg exposure-verify  --preset exposure_pilot.json --out verification/exposure_audit_report.txt
seqsfg exposure-run     --preset exposure_pilot.json --code P01
seqsfg exposure-analyze data/P01/session_01
```